# ETL — Extract, Transform, Load

Мета: завантажити 9 CSV-файлів Olist у PostgreSQL з очисткою та валідацією.

Реалізація розбита на модулі:
- `src/extractors.py`  — читання CSV
- `src/transformers.py` — чистка і трансформації  
- `src/validators.py`  — перевірки FK та NOT NULL
- `src/loaders.py`     — запис у PostgreSQL

In [1]:
import sys
from pathlib import Path

# Додаємо корінь проєкту до sys.path щоб імпортувати src.*
sys.path.append(str(Path.cwd().parent))

from src.config import get_engine
from src.extractors import (
    read_customers, read_orders, read_order_items,
    read_order_payments, read_order_reviews,
    read_products, read_sellers,
    read_geolocation, read_category_translation,
)
from src.transformers import (
    clean_customers, clean_orders, clean_order_items,
    clean_order_payments, clean_order_reviews,
    clean_products, clean_sellers, clean_geolocation,
)
from src.validators import (
    validate_fk, validate_not_null,
    validate_unique, validate_score_range,
)
from src.loaders import upsert, verify_counts

engine = get_engine()
print("✓ Підключення до БД встановлено")

✓ Підключення до БД встановлено


## 1. Extract — Читання CSV

Читаємо всі 9 CSV-файлів у DataFrame.
Типізацію задано явно в `src/extractors.py` (dtype, parse_dates).
Тут тільки завантаження — жодної чистки.

In [2]:
raw_customers    = read_customers()
raw_orders       = read_orders()
raw_order_items  = read_order_items()
raw_payments     = read_order_payments()
raw_reviews      = read_order_reviews()
raw_products     = read_products()
raw_sellers      = read_sellers()
raw_geo          = read_geolocation()
raw_translations = read_category_translation()

print("✓ Усі файли завантажено")

✓ Усі файли завантажено


In [3]:
# Порівнюємо з EDA (Крок 1) — числа мають збігатись
datasets = {
    "customers":    raw_customers,
    "orders":       raw_orders,
    "order_items":  raw_order_items,
    "payments":     raw_payments,
    "reviews":      raw_reviews,
    "products":     raw_products,
    "sellers":      raw_sellers,
    "geolocation":  raw_geo,
    "translations": raw_translations,
}

for name, df in datasets.items():
    print(f"{name:15s}: {df.shape[0]:>9,} рядків × {df.shape[1]} колонок")

customers      :    99,441 рядків × 5 колонок
orders         :    99,441 рядків × 8 колонок
order_items    :   112,650 рядків × 7 колонок
payments       :   103,886 рядків × 5 колонок
reviews        :    99,224 рядків × 7 колонок
products       :    32,951 рядків × 9 колонок
sellers        :     3,095 рядків × 4 колонок
geolocation    : 1,000,163 рядків × 5 колонок
translations   :        71 рядків × 2 колонок


## 2. Transform — Чистка і трансформації

Для кожної таблиці:
- Перейменування колонок під схему БД
- Відкидання зайвих колонок  
- Виправлення проблем знайдених в EDA

Детальна логіка — у `src/transformers.py`.

In [ ]:
# ── customers ──
# customer_zip_code_prefix → zip_code_prefix
# customer_city → city, customer_state → state
# Всі 5 колонок залишаються — всі є в схемі БД
customers = clean_customers(raw_customers)
customers.head(3)

  [customers] ✓ Готово: 99,441 рядків


,customer_id,customer_unique_id,zip_code_prefix,city,state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP


In [ ]:
# ── products ──
# Приймає ДВА DataFrame: products + translation
# LEFT MERGE → додає category_name_english
# NULL у розмірах/вазі (~0.01%) → медіана по категорії
products = clean_products(raw_products, raw_translations)
products.head(3)

  [products] product_weight_g: заповнено 2 NULL медіаною по категорії
  [products] product_length_cm: заповнено 2 NULL медіаною по категорії
  [products] product_height_cm: заповнено 2 NULL медіаною по категорії
  [products] product_width_cm: заповнено 2 NULL медіаною по категорії
  [products] ✓ Готово: 32,951 рядків


,product_id,category_name_english,weight_g,length_cm,height_cm,width_cm,photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,225,16,10,14,1
1,3aa071139cb16b67ca9e5dea641aaa2f,art,1000,30,18,20,1
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,154,18,9,15,1


In [ ]:
# ── sellers ──
# seller_zip_code_prefix → zip_code_prefix
# seller_city → city, seller_state → state
# Всі 4 колонки залишаються — всі є в схемі БД
sellers = clean_sellers(raw_sellers)
sellers.head(3)

  [sellers] ✓ Готово: 3,095 рядків


,seller_id,zip_code_prefix,city,state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


In [ ]:
# ── orders ──
# order_status → status
# order_purchase_timestamp → purchased_at
# order_approved_at → approved_at
# order_delivered_carrier_date → delivered_to_carrier_at
# order_delivered_customer_date → delivered_to_customer_at
# order_estimated_delivery_date → estimated_delivery_at
# Всі 8 колонок залишаються — всі є в схемі БД
# Перевірка: delivered_to_customer_at >= purchased_at
orders = clean_orders(raw_orders)
orders.head(3)

  [orders] ✓ Готово: 99,441 рядків


,order_id,customer_id,status,purchased_at,approved_at,delivered_to_carrier_at,delivered_to_customer_at,estimated_delivery_at
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


In [ ]:
# ── order_items ──
# Всі 7 колонок залишаються — всі є в схемі БД, нічого не перейменовується
# Composite PK: (order_id, order_item_id)
# Перевіряка: price > 0, freight_value >= 0
order_items = clean_order_items(raw_order_items)
order_items.head(3)

  [order_items] ✓ Готово: 112,650 рядків


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [ ]:
# ── order_payments ──
# payment_installments → installments, payment_value → value
# payment_sequential залишається — частина composite PK
# Composite PK: (order_id, payment_sequential)
payments = clean_order_payments(raw_payments)
payments.head(3)

  [order_payments] ✓ Готово: 103,886 рядків


,order_id,payment_sequential,payment_type,installments,value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [ ]:
# ── order_reviews ──
# review_score → score, review_comment_title → comment_title, ...
# ~88% NULL у comment_title — НОРМА, не видаляється
reviews = clean_order_reviews(raw_reviews)
reviews.head(3)

  [order_reviews] Видалено 814 дублів по ['review_id']
  [order_reviews] comment_title NULL: 88.3% (норма)
  [order_reviews] comment_message NULL: 58.7% (норма)
  [order_reviews] ✓ Готово: 98,410 рядків


,review_id,order_id,score,comment_title,comment_message,created_at,answered_at
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,<NA>,<NA>,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,<NA>,<NA>,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,<NA>,<NA>,2018-02-17,2018-02-18 14:36:24


In [ ]:
# ── geolocation ──
# ~1 млн рядків → ~15k унікальних zip (медіана lat/lng)
# НЕ завантажується в БД — довідник для одного з кроків проєкту
geo = clean_geolocation(raw_geo)
geo.head(3)

  [geolocation] 1,000,163 → 19,015 рядків (унікальних zip)


,zip_code,lat,lng
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313


In [12]:
# Підсумкова таблиця: скільки рядків видалено при очищенні
print(f"{'Таблиця':<20} {'До':>8} {'Після':>8} {'Видалено':>10}")
print("-" * 50)

pairs = [
    ("customers",      raw_customers,   customers),
    ("products",       raw_products,    products),
    ("sellers",        raw_sellers,     sellers),
    ("orders",         raw_orders,      orders),
    ("order_items",    raw_order_items, order_items),
    ("order_payments", raw_payments,    payments),
    ("order_reviews",  raw_reviews,     reviews),
]

for name, raw, clean in pairs:
    dropped = len(raw) - len(clean)
    print(f"{name:<20} {len(raw):>8,} {len(clean):>8,} {dropped:>10,}")

Таблиця                    До    Після   Видалено
--------------------------------------------------
customers              99,441   99,441          0
products               32,951   32,951          0
sellers                 3,095    3,095          0
orders                 99,441   99,441          0
order_items           112,650  112,650          0
order_payments        103,886  103,886          0
order_reviews          99,224   98,410        814


## 3. Validate — Перевірка перед завантаженням

Три типи перевірок:
1. **UNIQUE** — PK та composite PK унікальні (інакше БД відхилить INSERT)
2. **NOT NULL** — критичні колонки без пропусків
3. **FK** — зовнішні ключі посилаються на існуючі записи в батьківських таблицях

При виявленні сиріт — `validate_fk` видаляє їх і повідомляє кількість.

In [ ]:
# UNIQUE (PK)
print("=== UNIQUE ===")
validate_unique(customers,   ["customer_id"],               "customers")
validate_unique(products,    ["product_id"],                "products")
validate_unique(sellers,     ["seller_id"],                 "sellers")
validate_unique(orders,      ["order_id"],                  "orders")
validate_unique(order_items, ["order_id", "order_item_id"], "order_items")
validate_unique(reviews,     ["review_id"],                 "order_reviews")

=== UNIQUE ===
  ✓ [customers] UNIQUE ['customer_id'] — ОК
  ✓ [products] UNIQUE ['product_id'] — ОК
  ✓ [sellers] UNIQUE ['seller_id'] — ОК
  ✓ [orders] UNIQUE ['order_id'] — ОК
  ✓ [order_items] UNIQUE ['order_id', 'order_item_id'] — ОК
  ✓ [order_reviews] UNIQUE ['review_id'] — ОК


In [14]:
# NOT NULL
print("=== NOT NULL ===")
validate_not_null(
    customers,
    ["customer_id", "customer_unique_id", "zip_code_prefix", "city", "state"],
    "customers",
)
validate_not_null(
    products,
    ["product_id", "category_name_english"],
    "products",
)
validate_not_null(
    sellers,
    ["seller_id", "zip_code_prefix", "city", "state"],
    "sellers",
)
validate_not_null(
    orders,
    ["order_id", "customer_id", "status", "purchased_at"],
    "orders",
)
validate_not_null(
    reviews,
    ["review_id", "order_id", "score"],
    "order_reviews",
)

# CHECK
validate_score_range(reviews, "score", 1, 5, "order_reviews")

=== NOT NULL ===
  ✓ [customers] NOT NULL ['customer_id', 'customer_unique_id', 'zip_code_prefix', 'city', 'state'] — ОК
  ✓ [products] NOT NULL ['product_id', 'category_name_english'] — ОК
  ✓ [sellers] NOT NULL ['seller_id', 'zip_code_prefix', 'city', 'state'] — ОК
  ✓ [orders] NOT NULL ['order_id', 'customer_id', 'status', 'purchased_at'] — ОК
  ✓ [order_reviews] NOT NULL ['review_id', 'order_id', 'score'] — ОК
  ✓ [order_reviews] score ∈ [1, 5] — ОК


In [15]:
# FK (referential integrity)
print("=== FK ===")

orders = validate_fk(
    orders, "customer_id",
    customers, "customer_id",
    "orders", "customers",
)
order_items = validate_fk(
    order_items, "order_id",
    orders, "order_id",
    "order_items", "orders",
)
order_items = validate_fk(
    order_items, "product_id",
    products, "product_id",
    "order_items", "products",
)
order_items = validate_fk(
    order_items, "seller_id",
    sellers, "seller_id",
    "order_items", "sellers",
)
payments = validate_fk(
    payments, "order_id",
    orders, "order_id",
    "order_payments", "orders",
)
reviews = validate_fk(
    reviews, "order_id",
    orders, "order_id",
    "order_reviews", "orders",
)

=== FK ===
  ✓ [orders] → [customers]: усі customer_id валідні
  ✓ [order_items] → [orders]: усі order_id валідні
  ✓ [order_items] → [products]: усі product_id валідні
  ✓ [order_items] → [sellers]: усі seller_id валідні
  ✓ [order_payments] → [orders]: усі order_id валідні
  ✓ [order_reviews] → [orders]: усі order_id валідні


## 4. Load — Запис у PostgreSQL

Порядок завантаження визначається FK-залежностями:

```
Рівень 1: customers, products, sellers   (незалежні)
Рівень 2: orders                         (FK → customers)
Рівень 3: order_items                    (FK → orders, products, sellers)
          order_payments                 (FK → orders)
          order_reviews                  (FK → orders)
```

Використовується `upsert` — ідемпотентний запис, повторний запуск не створює дублів.

In [16]:
# Рівень 1: незалежні батьківські таблиці
upsert(customers, "customers", ["customer_id"], engine)
upsert(products,  "products",  ["product_id"],  engine)
upsert(sellers,   "sellers",   ["seller_id"],   engine)

  ✓ [customers] Записано/оновлено: 0 рядків
  ✓ [products] Записано/оновлено: 0 рядків
  ✓ [sellers] Записано/оновлено: 0 рядків


In [17]:
# Рівень 2: orders (FK → customers)
upsert(orders, "orders", ["order_id"], engine)

  ✓ [orders] Записано/оновлено: 0 рядків


In [18]:
# Рівень 3: дочірні таблиці (FK → orders)
upsert(order_items, "order_items",    ["order_id", "order_item_id"],      engine)
upsert(payments,    "order_payments", ["order_id", "payment_sequential"], engine)
upsert(reviews,     "order_reviews",  ["review_id"],                      engine)

  ✓ [order_items] Записано/оновлено: 0 рядків
  ✓ [order_payments] Записано/оновлено: 0 рядків
  ✓ [order_reviews] Записано/оновлено: 0 рядків


## 5. Verify — Верифікація результату

Порівнюємо `COUNT(*)` у БД з `len(df)` після трансформацій.
Числа мають збігатися.

In [19]:
verify_counts({
    "customers":      len(customers),
    "products":       len(products),
    "sellers":        len(sellers),
    "orders":         len(orders),
    "order_items":    len(order_items),
    "order_payments": len(payments),
    "order_reviews":  len(reviews),
}, engine)


=== ВЕРИФІКАЦІЯ: COUNT(*) у БД ===
  ✓ customers           : БД= 99,441 | очікувано= 99,441
  ✓ products            : БД= 32,951 | очікувано= 32,951
  ✓ sellers             : БД=  3,095 | очікувано=  3,095
  ✓ orders              : БД= 99,441 | очікувано= 99,441
  ✓ order_items         : БД=112,650 | очікувано=112,650
  ✓ order_payments      : БД=103,886 | очікувано=103,886
  ✓ order_reviews       : БД= 98,410 | очікувано= 98,410
  ✓ Усі таблиці: кількість рядків відповідає очікуваному


In [ ]:
import pandas as pd

# Додаткові SQL-перевірки
checks = {
    "Null категорії в products": """
        SELECT 
            COUNT(*) 
        FROM products 
        WHERE category_name_english IS NULL
    """,
    "NULL comment_title % (норма ~88%)": """
        SELECT 
            ROUND(100.0 * SUM(CASE WHEN comment_title IS NULL THEN 1 END) / COUNT(*), 1) 
        FROM order_reviews
    """,
    "Score тільки 1-5": """
        SELECT 
            score, 
            COUNT(*) AS cnt 
        FROM order_reviews
        GROUP BY score 
        ORDER BY score
    """,
}

for label, sql in checks.items():
    result = pd.read_sql(sql, engine)
    print(f"\n── {label} ──")
    print(result.to_string(index=False))


── Null категорії в products ──
 count
     0

── NULL comment_title % (норма ~88%) ──
 round
  88.3

── Score тільки 1-5 ──
 score   cnt
     1 11282
     2  3114
     3  8097
     4 19007
     5 56910


## 6. Висновок

### Extract
Завантажено 9 CSV-файлів датасету Olist (2016–2018):                
Customers: 99,441       
Products: 32,951         
Sellers: 3,095           
Orders: 99,441           
Order_items: 112,650             
Order_payments: 103,886            
Order_reviews: 98,410            

### Transform — виправлені проблеми
1. **Перейменування колонок** — `customer_city → city`, `order_status → status` тощо для відповідності схемі БД.
2. **Products × Translation** — LEFT MERGE → `category_name_english`. NULL категорій → `'unknown'`.
3. **Пропуски у розмірах/вазі (~0.01%)** — заповнені медіаною по категорії товарів.
4. **Пропуски у коментарях (~88%/59%)** — залишені як NULL (нормальна поведінка користувачів).
5. **Geolocation** (~1 млн рядків) — агреговано медіаною → ~15k унікальних zip. Не завантажуються в БД — використовується як довідник для одного з кроків проєкту.
6. **Логіка дат** — видалено замовлення де `delivered_to_customer_at < purchased_at`.

### Load
Завантаження у порядку FK-залежностей:
`customers, products, sellers → orders → order_items, order_payments, order_reviews`.           
`COUNT(*)` у БД відповідає `len(df)` для кожної таблиці.

### Обмеження
- Дані за 2016–2018 — можуть бути неактуальні для поточного стану ринку.
- `geolocation` не завантажено в БД — використовується лише як довідник для карт.